# Copernicus Marine Service: Access Model Output
* Need to have an account on Copernicus Marine
* The ~/.netrc file contains the account credentials: your login and password for [Copernicus Marine](https://marine.copernicus.eu/)
```
$ more ~/.netrc
machine auth.marine.copernicus.eu
  login xxxxxxxxxx 
  password xxxxxxxxxxxxx
```

In [ ]:
import copernicusmarine
import os

In [ ]:
%%time 
ds = copernicusmarine.open_dataset(dataset_id='cmems_mod_med_phy-temp_my_4.2km_P1D-m')

In [ ]:
ds

In [ ]:
#here you create a subset in which you select the latitude based on the values (sel) and the depth based on the index (isel)
ds_subset = ds['thetao'].sel(latitude=slice(30, 47), longitude=slice(-6, 42)).isel(depth=0) 
ds_subset

In [ ]:
#ds_subset.groupby('time.dayofyear').mean('time')
ds_subset.groupby('time.year').mean('time')

In [ ]:
cluster_type = 'Coiled'
#cluster_type = 'Gateway'

In [ ]:
if cluster_type == 'Coiled':
    import coiled
    cluster = coiled.Cluster(
        region="eu-central-1",
        arm=True,   # run on ARM to save energy & cost
        worker_vm_types=["t4g.large"],  # cheap, small ARM instances, 2cpus, 2GB RAM
        n_workers=30, 
        name = "Emma",
        wait_for_workers=False,
        compute_purchase_option="spot_with_fallback",
        software='protocoast-notebook-arm',  # Conda environment name
        workspace='esip-lab',
        timeout=180   # leave cluster running for 3 min in case we want to use it again
    )

    client = cluster.get_client()

In [ ]:
%%time
if cluster_type == 'Gateway':   # Protocoast JupyterHub
    from dask_gateway import Gateway

    gateway = Gateway()  # instantiate Dask gateway 

    # Cluster options on Nebari 
    options = gateway.cluster_options()
    options.image = os.environ['JUPYTER_IMAGE']

    # Create a Dask Gateway cluster
    cluster = gateway.new_cluster(options)
    # Scale the cluster
    cluster.adapt(minimum=2, maximum=24)
    # Get the Dask client for the Dask Gateway cluster
    client = cluster.get_client()

In [ ]:
client

In [ ]:
#da = ds_subset.groupby('time.dayofyear').mean('time').compute()
da = ds_subset.groupby('time.year').mean('time').compute()

In [ ]:
import hvplot.xarray
import panel as pn
pn.extension()

In [ ]:
da.hvplot(x='longitude', y='latitude', rasterize=True, geo=True, frame_width=500, cmap='turbo', tiles='OSM')

## TIME SERIES 
Here I'm pulling data temperatures for the whole month of August plotting the daily mean with a bar that allows to go trhough the different days

In [ ]:
%%time
# Load all August days at surface depth across all years (1987–2025)
da_all_aug = (
    ds['thetao']
    .sel(depth=0, method='nearest')
    .isel(time=(ds.time.dt.month == 8).values)
    .load()
)
# Per-pixel mean for each calendar day of August, averaged across all years
da_aug_clim = da_all_aug.groupby('time.day').mean('time')

In [ ]:
da_aug_clim.hvplot(
    x='longitude', y='latitude',
    groupby='day',
    rasterize=True, geo=True,
    frame_width=600,
    cmap='turbo',
    clim=(da_aug_clim.min().item(), da_aug_clim.max().item()),
    tiles='OSM',
    title='August Daily Mean Surface Temperature — Climatology 1987–2025',
    widget_type='scrubber',
    widget_location='bottom',
)

In [ ]:
cluster.shutdown()